In [1]:
# 1) Setup & Imports
import os
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
 )
from sklearn.model_selection import GroupShuffleSplit, GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils import resample

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
PREPROCESSED_DIR = Path(r"C:\Users\buck\Napplee\StressClassification\data\preprocessed")
DATA_ROOT = PREPROCESSED_DIR.parent
CONCATENATED_DIR = DATA_ROOT / "concatenated\\day_scenarios"
FEATURE_DIR = Path(r"C:\Users\buck\Napplee\StressClassification\data\features")
FEATURE_FILE = FEATURE_DIR / "features_hrv.csv"
ARTIFACTS_DIR = Path("artifacts")
REPORTS_DIR = Path("reports")

for p in [FEATURE_DIR, ARTIFACTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Preprocessed directory: {PREPROCESSED_DIR}")
print(f"Concatenated directory: {CONCATENATED_DIR}")
print(f"Feature file output: {FEATURE_FILE}")
print(f"Artifacts directory: {ARTIFACTS_DIR.resolve()}")

Preprocessed directory: C:\Users\buck\Napplee\StressClassification\data\preprocessed
Concatenated directory: C:\Users\buck\Napplee\StressClassification\data\concatenated\day_scenarios
Feature file output: C:\Users\buck\Napplee\StressClassification\data\features\features_hrv.csv
Artifacts directory: C:\Users\buck\Napplee\StressClassification\artifacts


In [ ]:
# --- Extract features from all 24h CSV files in folder, windowed, label by majority vote, save to features_hrv.csv ---
from pathlib import Path
import pandas as pd
import numpy as np

DATA_24H_DIR = Path(r'C:/Users/buck/Napplee/StressClassification/data/concatenated/day_scenarios')
FEATURE_FILE = Path(r'C:/Users/buck/Napplee/StressClassification/data/features/features_hrv.csv')
WINDOW_SEC = 1800  # 30 minutes

feature_rows = []
csv_files = list(DATA_24H_DIR.glob('*.csv'))
print(f'Tìm thấy {len(csv_files)} file CSV trong thư mục day_scenarios')
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    if not {'Time', 'Voltage', 'Peak', 'label'}.issubset(df.columns):
        print(f'Bỏ qua {csv_file.name} vì thiếu cột cần thiết!')
        continue
    df = df.sort_values('Time').reset_index(drop=True)
    min_time = df['Time'].min()
    max_time = df['Time'].max()
    n_windows = int(np.ceil((max_time - min_time) / WINDOW_SEC))
    for i in range(n_windows):
        start = min_time + i * WINDOW_SEC
        end = start + WINDOW_SEC
        win_df = df[(df['Time'] >= start) & (df['Time'] < end)]
        if win_df.empty:
            continue
        label_val = win_df['label'].mode().iloc[0] if not win_df['label'].isnull().all() else np.nan
        row = {
            'source_file': csv_file.name,
            'window_id': i,
            'start_time': start,
            'end_time': end,
            'label': label_val,
            'mean_voltage': win_df['Voltage'].mean(),
            'std_voltage': win_df['Voltage'].std(),
            'n_peak': (win_df['Peak'] == 3).sum(),
            'duration_sec': win_df['Time'].max() - win_df['Time'].min(),
            # Add more features as needed
        }
        feature_rows.append(row)

features_df = pd.DataFrame(feature_rows)
features_df.to_csv(FEATURE_FILE, index=False)
print(f'Đã lưu {len(features_df)} window vào {FEATURE_FILE}')
print('Phân bố label:')
print(features_df['label'].value_counts())
display(features_df.head())


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ---- ĐƯỜNG DẪN ----
RAW_FOLDER = Path(r'C:/Users/buck/Napplee/StressClassification/data/concatenated/day_scenarios')   # <-- THAY BẰNG THƯ MỤC CỦA BẠN!
FEATURE_FILE = Path(r'C:/Users/buck/Napplee/StressClassification/data/features/features_hrv.csv')

WINDOW_SEC = 1800.0  # 30 phút
MIN_RR_COUNT = 5
MIN_WINDOW_DURATION_SEC = 20.0
MIN_HR_BPM = 40.0
MAX_HR_BPM = 190.0

def clean_waveform_group(g):
    g = g.sort_values("Time").drop_duplicates("Time")
    g = g[g["Time"].notna() & (g["Time"] >= 0)]
    g = g[g["Time"].diff().fillna(1) > 0]
    g["Peak"] = pd.to_numeric(g["Peak"], errors="coerce")
    v = pd.to_numeric(g["Voltage"], errors="coerce")
    q_low, q_high = np.nanquantile(v, [0.005, 0.995])
    g["Voltage"] = v.clip(lower=q_low, upper=q_high)
    return g.dropna(subset=["Time", "Voltage", "Peak"])

def safe_skew(x):
    return float(pd.Series(x).skew()) if len(x) >= 3 else np.nan

def safe_kurt(x):
    return float(pd.Series(x).kurt()) if len(x) >= 4 else np.nan

def rr_frequency_features(rr_ms):
    if len(rr_ms) < 4: return (np.nan, np.nan, np.nan, np.nan)
    rr_s = rr_ms / 1000.0
    t = np.cumsum(rr_s) - rr_s[0]
    if t[-1] <= 0: return (np.nan, np.nan, np.nan, np.nan)
    fs = 4.0
    t_uniform = np.arange(0, t[-1], 1 / fs)
    if len(t_uniform) < 8: return (np.nan, np.nan, np.nan, np.nan)
    rr_interp = np.interp(t_uniform, t, rr_ms)
    rr_detrended = rr_interp - np.nanmean(rr_interp)
    fft_vals = np.fft.rfft(rr_detrended)
    freqs = np.fft.rfftfreq(len(rr_detrended), d=1 / fs)
    psd = (np.abs(fft_vals) ** 2) / max(len(rr_detrended), 1)
    lf_mask = (freqs >= 0.04) & (freqs < 0.15)
    hf_mask = (freqs >= 0.15) & (freqs <= 0.40)
    total_mask = (freqs >= 0.04) & (freqs <= 0.40)
    lf_power = float(np.trapezoid(psd[lf_mask], freqs[lf_mask])) if np.any(lf_mask) else np.nan
    hf_power = float(np.trapezoid(psd[hf_mask], freqs[hf_mask])) if np.any(hf_mask) else np.nan
    total_power = float(np.trapezoid(psd[total_mask], freqs[total_mask])) if np.any(total_mask) else np.nan
    lf_hf_ratio = (lf_power / hf_power) if (pd.notna(lf_power) and pd.notna(hf_power) and hf_power > 0) else np.nan
    return lf_power, hf_power, lf_hf_ratio, total_power

def extract_features_from_window(g, window_id=0, source_file="raw_file", window_start=None, window_end=None):
    g = clean_waveform_group(g)
    duration_sec = float(g["Time"].max() - g["Time"].min()) if len(g) else 0.0
    if len(g) < 20 or duration_sec < MIN_WINDOW_DURATION_SEC:
        return None
    peak_times = g.loc[g["Peak"] == 3, "Time"].values
    if len(peak_times) < 4:
        return None
    rr_raw = np.diff(peak_times) * 1000.0  # ms
    rr_ms = rr_raw[(rr_raw >= 300.0) & (rr_raw <= 2000.0)]
    rr_valid_ratio = float(len(rr_ms) / max(len(rr_raw), 1))
    if len(rr_ms) < MIN_RR_COUNT:
        return None
    rmssd = np.sqrt(np.mean(np.diff(rr_ms) ** 2)) if len(rr_ms) >= 2 else np.nan
    pnn50 = float(np.mean(np.abs(np.diff(rr_ms)) > 50.0) * 100.0) if len(rr_ms) >= 2 else np.nan
    mean_rr_ms = float(np.mean(rr_ms))
    sdnn_ms = float(np.std(rr_ms, ddof=1)) if len(rr_ms) > 1 else np.nan
    hr_bpm = 60000.0 / mean_rr_ms if mean_rr_ms > 0 else np.nan
    n_peaks = int(np.sum(g["Peak"] == 3))
    peak_rate = n_peaks / duration_sec if duration_sec > 0 else np.nan
    lf_power, hf_power, lf_hf_ratio, total_power = rr_frequency_features(rr_ms)
    label_val = g["label"].mode().iloc[0] if "label" in g.columns and not g["label"].isnull().all() else np.nan
    return {
        "source_file": source_file,
        "window_id": window_id,
        "window_start": window_start,
        "window_end": window_end,
        "label": label_val,
        "mean_rr_ms": mean_rr_ms,
        "median_rr_ms": float(np.median(rr_ms)),
        "sdnn_ms": sdnn_ms,
        "rmssd_ms": float(rmssd),
        "pnn50": float(pnn50),
        "heart_rate_bpm": float(hr_bpm),
        "n_peaks": n_peaks,
        "duration_sec": duration_sec,
        "peak_rate_per_sec": float(peak_rate),
        "rr_valid_ratio": rr_valid_ratio,
        "voltage_mean": float(g["Voltage"].mean()),
        "voltage_std": float(g["Voltage"].std(ddof=1)) if len(g) > 1 else np.nan,
        "voltage_min": float(g["Voltage"].min()),
        "voltage_max": float(g["Voltage"].max()),
        "voltage_skew": safe_skew(g["Voltage"].values),
        "voltage_kurtosis": safe_kurt(g["Voltage"].values),
        "lf_power": lf_power,
        "hf_power": hf_power,
        "lf_hf_ratio": lf_hf_ratio,
        "total_power": total_power,
    }

# ---- MAIN ----

all_feature_rows = []

for csv_path in RAW_FOLDER.glob("*.csv"):
    try:
        df = pd.read_csv(csv_path)
        if not all(col in df.columns for col in ["Time", "Voltage", "Peak"]):
            print(f"Bỏ qua {csv_path.name} vì thiếu cột bắt buộc!")
            continue
        df = df.sort_values("Time").reset_index(drop=True)
        min_time = df["Time"].min()
        max_time = df["Time"].max()
        n_windows = int(np.ceil((max_time - min_time) / WINDOW_SEC))
        for i in range(n_windows):
            start = min_time + i * WINDOW_SEC
            end = start + WINDOW_SEC
            win_df = df[(df["Time"] >= start) & (df["Time"] < end)].copy()
            if win_df.empty:
                continue
            row = extract_features_from_window(
                win_df,
                window_id=i,
                source_file=csv_path.name,
                window_start=start,
                window_end=end
            )
            if row is not None:
                all_feature_rows.append(row)
    except Exception as ex:
        print(f"Lỗi đọc {csv_path}: {ex}")

features_df = pd.DataFrame(all_feature_rows)
if features_df.empty:
    raise ValueError("Không trích xuất được feature từ bất kỳ file nào!")

# ---- (OPTIONAL) Lọc chất lượng đầu ra ----
features_df = features_df[
    (features_df["duration_sec"] >= MIN_WINDOW_DURATION_SEC) &
    (features_df["heart_rate_bpm"].between(MIN_HR_BPM, MAX_HR_BPM))
]
features_df = features_df.reset_index(drop=True)

features_df.to_csv(FEATURE_FILE, index=False)
print(f"Đã trích xuất và lưu {len(features_df)} window vào {FEATURE_FILE}")
print("Phân bố label:")
print(features_df["label"].value_counts(dropna=False))
print(features_df.head())

Đã trích xuất và lưu 480 window vào C:\Users\buck\Napplee\StressClassification\data\features\features_hrv.csv
Phân bố label:
label
0    218
1    124
2     76
3     62
Name: count, dtype: int64
                   source_file  window_id  window_start  window_end  label  \
0  day_custom_segments_001.csv          0           0.0      1800.0      0   
1  day_custom_segments_001.csv          1        1800.0      3600.0      0   
2  day_custom_segments_001.csv          2        3600.0      5400.0      0   
3  day_custom_segments_001.csv          3        5400.0      7200.0      0   
4  day_custom_segments_001.csv          4        7200.0      9000.0      0   

   mean_rr_ms  median_rr_ms     sdnn_ms   rmssd_ms      pnn50  ...  \
0  692.462846     683.59375   92.076921  61.924494  32.371055  ...   
1  674.935824     660.15625   96.415759  61.244377  28.893058  ...   
2  722.557417     722.65625  101.599960  68.132995  37.967055  ...   
3  700.805284     691.40625   90.059851  64.607337  34.008

## Test nhanh cách mã hoá Pointcaré

Phần này mô phỏng đúng ý tưởng trong bài mẫu: lấy các đỉnh R, suy ra dãy khoảng R-R, rồi rời rạc hoá thành ma trận Pointcaré 30 × 30 với miền giá trị từ 400 ms đến 1400 ms.

Khác với pipeline hiện tại của notebook, đoạn test này không trích đặc trưng thống kê như mean, std hay LF/HF. Mục đích của nó là giúp bạn kiểm tra nhanh xem dữ liệu ECG của mình có tạo ra được RR interval hợp lệ không, và nhìn trực quan ma trận đầu vào trước khi đưa vào mô hình.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAW_FOLDER = Path(r"C:/Users/buck/Napplee/StressClassification/data/concatenated/day_scenarios")
WINDOW_SEC_TEST = 120.0
GRID_SIZE = 30
RR_MIN_MS = 400.0
RR_MAX_MS = 1400.0


def extract_rr_from_window(window_df):
    window_df = window_df.sort_values("Time").drop_duplicates("Time")
    window_df = window_df[window_df["Time"].notna() & window_df["Peak"].notna()]
    peak_times = pd.to_numeric(window_df.loc[window_df["Peak"] == 3, "Time"], errors="coerce").dropna().values
    if len(peak_times) < 2:
        return np.array([], dtype=float)
    rr_ms = np.diff(peak_times) * 1000.0
    rr_ms = rr_ms[(rr_ms >= RR_MIN_MS) & (rr_ms <= RR_MAX_MS)]
    return rr_ms


def build_pointcare_matrix(rr_ms, grid_size=30, rr_min_ms=400.0, rr_max_ms=1400.0):
    rr_ms = np.asarray(rr_ms, dtype=float)
    rr_ms = rr_ms[np.isfinite(rr_ms)]
    rr_ms = rr_ms[(rr_ms >= rr_min_ms) & (rr_ms <= rr_max_ms)]

    matrix = np.zeros((grid_size, grid_size), dtype=np.uint8)
    if len(rr_ms) < 2:
        return matrix, 0

    bins = np.linspace(rr_min_ms, rr_max_ms, grid_size + 1)
    x_idx = np.digitize(rr_ms[:-1], bins, right=False) - 1
    y_idx = np.digitize(rr_ms[1:], bins, right=False) - 1
    valid_mask = (
        (x_idx >= 0) & (x_idx < grid_size) &
        (y_idx >= 0) & (y_idx < grid_size)
    )
    matrix[y_idx[valid_mask], x_idx[valid_mask]] = 1
    return matrix, int(valid_mask.sum())


def encode_state_row(label_value):
    if pd.isna(label_value):
        return np.nan

    label_text = str(label_value).strip().lower()
    rest_tokens = {"0", "rest", "nghi", "nghỉ", "nghỉ ngơi", "normal", "relax", "0.0"}
    work_tokens = {"1", "work", "lam viec", "làm việc", "stress", "1.0"}

    if label_text in rest_tokens:
        return 0.1
    if label_text in work_tokens:
        return 0.5

    try:
        label_num = float(label_value)
        if label_num <= 0:
            return 0.1
        if label_num >= 1:
            return 0.5
    except Exception:
        pass

    return np.nan


def find_first_valid_window(csv_path, window_sec=120.0):
    df = pd.read_csv(csv_path)
    if not {"Time", "Voltage", "Peak"}.issubset(df.columns):
        return None, None

    df = df.sort_values("Time").reset_index(drop=True)
    min_time = df["Time"].min()
    max_time = df["Time"].max()
    n_windows = int(np.ceil((max_time - min_time) / window_sec))

    for window_id in range(n_windows):
        start = min_time + window_id * window_sec
        end = start + window_sec
        window_df = df[(df["Time"] >= start) & (df["Time"] < end)].copy()
        if window_df.empty:
            continue

        rr_ms = extract_rr_from_window(window_df)
        if len(rr_ms) >= 2:
            return window_df, rr_ms

    return None, None


csv_files = sorted(RAW_FOLDER.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Không tìm thấy file CSV trong {RAW_FOLDER}")

sample_file = csv_files[0]
sample_window_df, sample_rr_ms = find_first_valid_window(sample_file, window_sec=WINDOW_SEC_TEST)
if sample_window_df is None:
    raise ValueError(f"Không tìm được window hợp lệ trong file {sample_file.name}")

pointcare_matrix, n_points = build_pointcare_matrix(
    sample_rr_ms,
    grid_size=GRID_SIZE,
    rr_min_ms=RR_MIN_MS,
    rr_max_ms=RR_MAX_MS,
)

label_value = sample_window_df["label"].mode().iloc[0] if "label" in sample_window_df.columns and not sample_window_df["label"].isnull().all() else np.nan
state_value = encode_state_row(label_value)

pointcare_with_state = np.vstack([
    pointcare_matrix.astype(float),
    np.zeros((1, GRID_SIZE), dtype=float),
])
if pd.notna(state_value):
    pointcare_with_state[-1, 0] = state_value

print(f"File kiểm tra: {sample_file.name}")
print(f"Số RR hợp lệ: {len(sample_rr_ms)}")
print(f"Số điểm Pointcaré được ánh xạ vào lưới: {n_points}")
print(f"Nhãn window: {label_value}")
print(f"Giá trị trạng thái mã hoá: {state_value}")
print(f"Kích thước ma trận Pointcaré: {pointcare_matrix.shape}")
print(f"Kích thước sau khi thêm hàng trạng thái: {pointcare_with_state.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(pointcare_matrix, cmap="Blues", origin="lower", interpolation="nearest")
axes[0].set_title("Pointcaré 30 × 30")
axes[0].set_xlabel("RR(n) bin")
axes[0].set_ylabel("RR(n+1) bin")

axes[1].imshow(pointcare_with_state, cmap="Blues", origin="lower", interpolation="nearest")
axes[1].set_title("Pointcaré + hàng trạng thái")
axes[1].set_xlabel("Cột")
axes[1].set_ylabel("Hàng")

plt.tight_layout()
plt.show()

## So sánh 2 cách: (A) Feature engineering vs (B) Pointcaré

### (A) Code hiện tại của bạn đang làm gì?
Trong notebook này, pipeline chính đang theo hướng **feature engineering từ ECG/Peak → RR interval → đặc trưng số**:
- Lấy các đỉnh R (đang có sẵn trong cột `Peak == 3`) để tính dãy RR (ms).
- Lọc RR “hợp lý” theo ngưỡng (ví dụ 300–2000 ms) để bỏ nhịp sai.
- Trích đặc trưng miền thời gian: `mean_rr_ms`, `sdnn`, `rmssd`, `pnn50`, suy ra `heart_rate_bpm`, …
- Trích đặc trưng miền tần số (xấp xỉ): nội suy RR → FFT → công suất LF/HF → `lf_hf_ratio`.
- (Tuỳ đoạn code) cộng thêm thống kê tín hiệu như `Voltage` (mean/std/min/max/skew/kurtosis), số peak, tỷ lệ RR hợp lệ…

=> Kết quả là **mỗi cửa sổ thời gian** (window) được biến thành **1 vector số** để đưa vào các mô hình sklearn (LogReg/SVM/RF/KNN,…).

### (B) Pointcaré (bài mẫu) khác gì?
Pointcaré không tạo vector đặc trưng thủ công, mà biến chuỗi RR thành **“ảnh” 2D**:
- Tạo các cặp điểm $(RR_n, RR_{n+1})$.
- Rời rạc hoá vào lưới 30×30 trong miền 400–1400 ms.
- Mỗi ô có điểm thì gán 1, không có thì 0.

Lưu ý quan trọng: bài mẫu thêm **hàng trạng thái (nghỉ/làm việc)** là *thông tin ngữ cảnh* biết trước. Nếu bạn dùng chính `label` làm “trạng thái” đầu vào thì sẽ bị **leak (rò rỉ nhãn)**. Vì vậy phần benchmark bên dưới **không dùng hàng trạng thái** (chỉ dùng Pointcaré 30×30).

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# ---------- CONFIG ----------
RAW_FOLDER = Path(r"C:/Users/buck/Napplee/StressClassification/data/concatenated/day_scenarios")
WINDOW_SEC = 120.0  # 2 phút (giống bài mẫu)
GRID_SIZE = 30
RR_MIN_MS_POINTCARE = 400.0
RR_MAX_MS_POINTCARE = 1400.0


# ---------- Helpers: RR + Pointcaré ----------
def _clean_and_get_peaks(df):
    df = df.sort_values("Time").drop_duplicates("Time")
    df = df[df["Time"].notna() & (df["Time"] >= 0)]
    df["Peak"] = pd.to_numeric(df["Peak"], errors="coerce")
    return df


def extract_rr_ms_from_window(window_df, rr_min_ms=300.0, rr_max_ms=2000.0):
    window_df = _clean_and_get_peaks(window_df)
    peak_times = pd.to_numeric(window_df.loc[window_df["Peak"] == 3, "Time"], errors="coerce").dropna().values
    if len(peak_times) < 2:
        return np.array([], dtype=float)
    rr_raw = np.diff(peak_times) * 1000.0
    rr_ms = rr_raw[(rr_raw >= rr_min_ms) & (rr_raw <= rr_max_ms)]
    return rr_ms


def build_pointcare_matrix(rr_ms, grid_size=30, rr_min_ms=400.0, rr_max_ms=1400.0):
    rr_ms = np.asarray(rr_ms, dtype=float)
    rr_ms = rr_ms[np.isfinite(rr_ms)]
    rr_ms = rr_ms[(rr_ms >= rr_min_ms) & (rr_ms <= rr_max_ms)]

    matrix = np.zeros((grid_size, grid_size), dtype=np.uint8)
    if len(rr_ms) < 2:
        return matrix, 0

    bins = np.linspace(rr_min_ms, rr_max_ms, grid_size + 1)
    x_idx = np.digitize(rr_ms[:-1], bins, right=False) - 1
    y_idx = np.digitize(rr_ms[1:], bins, right=False) - 1
    valid = (x_idx >= 0) & (x_idx < grid_size) & (y_idx >= 0) & (y_idx < grid_size)
    matrix[y_idx[valid], x_idx[valid]] = 1
    return matrix, int(valid.sum())


def find_first_valid_window(csv_path, window_sec=120.0):
    df = pd.read_csv(csv_path)
    required = {"Time", "Voltage", "Peak"}
    if not required.issubset(df.columns):
        return None, None, None

    df = df.sort_values("Time").reset_index(drop=True)
    min_time = df["Time"].min()
    max_time = df["Time"].max()
    n_windows = int(np.ceil((max_time - min_time) / window_sec))

    for window_id in range(n_windows):
        start = min_time + window_id * window_sec
        end = start + window_sec
        win = df[(df["Time"] >= start) & (df["Time"] < end)].copy()
        if win.empty:
            continue
        rr_ms = extract_rr_ms_from_window(win)
        if len(rr_ms) >= 10:
            return win, rr_ms, window_id

    return None, None, None


# ---------- Helpers: Feature engineering (fallback) ----------
def rr_frequency_features(rr_ms):
    # Giữ đơn giản & gần với code bạn đang có (nội suy RR, FFT, tính LF/HF)
    rr_ms = np.asarray(rr_ms, dtype=float)
    if len(rr_ms) < 4:
        return np.nan, np.nan, np.nan

    rr_s = rr_ms / 1000.0
    t = np.cumsum(rr_s) - rr_s[0]
    if t[-1] <= 0:
        return np.nan, np.nan, np.nan

    fs = 4.0
    t_uniform = np.arange(0, t[-1], 1 / fs)
    if len(t_uniform) < 8:
        return np.nan, np.nan, np.nan

    rr_interp = np.interp(t_uniform, t, rr_ms)
    rr_interp = rr_interp - np.nanmean(rr_interp)

    fft_vals = np.fft.rfft(rr_interp)
    freqs = np.fft.rfftfreq(len(rr_interp), d=1 / fs)
    psd = (np.abs(fft_vals) ** 2) / max(len(rr_interp), 1)

    lf_mask = (freqs >= 0.04) & (freqs < 0.15)
    hf_mask = (freqs >= 0.15) & (freqs <= 0.40)

    lf_power = float(np.trapezoid(psd[lf_mask], freqs[lf_mask])) if np.any(lf_mask) else np.nan
    hf_power = float(np.trapezoid(psd[hf_mask], freqs[hf_mask])) if np.any(hf_mask) else np.nan
    ratio = (lf_power / hf_power) if (pd.notna(lf_power) and pd.notna(hf_power) and hf_power > 0) else np.nan
    return lf_power, hf_power, ratio


def feature_vector_from_window(window_df):
    # Ưu tiên gọi đúng hàm extract_features_from_window nếu bạn đã chạy Cell 3.
    if "extract_features_from_window" in globals():
        row = extract_features_from_window(window_df, window_id=0, source_file="sample")
        return row

    # Fallback tối giản (để vẫn chạy được benchmark)
    rr_ms = extract_rr_ms_from_window(window_df)
    if len(rr_ms) < 5:
        return None

    rmssd = float(np.sqrt(np.mean(np.diff(rr_ms) ** 2))) if len(rr_ms) >= 2 else np.nan
    pnn50 = float(np.mean(np.abs(np.diff(rr_ms)) > 50.0) * 100.0) if len(rr_ms) >= 2 else np.nan
    mean_rr = float(np.mean(rr_ms))
    sdnn = float(np.std(rr_ms, ddof=1)) if len(rr_ms) > 1 else np.nan
    hr = 60000.0 / mean_rr if mean_rr > 0 else np.nan
    lf, hf, lf_hf = rr_frequency_features(rr_ms)

    label_val = window_df["label"].mode().iloc[0] if ("label" in window_df.columns and not window_df["label"].isnull().all()) else np.nan

    return {
        "label": label_val,
        "mean_rr_ms": mean_rr,
        "sdnn_ms": sdnn,
        "rmssd_ms": rmssd,
        "pnn50": pnn50,
        "heart_rate_bpm": float(hr),
        "lf_power": float(lf) if pd.notna(lf) else np.nan,
        "hf_power": float(hf) if pd.notna(hf) else np.nan,
        "lf_hf_ratio": float(lf_hf) if pd.notna(lf_hf) else np.nan,
    }


# ---------- 1 window: in ra khác biệt ----------
csv_files = sorted(RAW_FOLDER.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Không có CSV trong {RAW_FOLDER}")

sample_file = csv_files[0]
win_df, rr_ms, window_id = find_first_valid_window(sample_file, window_sec=WINDOW_SEC)
if win_df is None:
    raise ValueError(f"Không tìm thấy window hợp lệ trong {sample_file.name}")

feat_row = feature_vector_from_window(win_df)
pointcare_matrix, n_points = build_pointcare_matrix(
    rr_ms,
    grid_size=GRID_SIZE,
    rr_min_ms=RR_MIN_MS_POINTCARE,
    rr_max_ms=RR_MAX_MS_POINTCARE,
)

print("=== SAMPLE WINDOW ===")
print("file:", sample_file.name)
print("window_id:", window_id)
print("rr_count (after 300–2000ms filter):", len(rr_ms))
print("pointcare nonzero cells:", int(pointcare_matrix.sum()))
print("pointcare mapped points:", n_points)

print("\n--- Feature engineering output (1 row) ---")
print(pd.Series(feat_row).sort_index() if feat_row is not None else "(None)")

print("\n--- Pointcaré matrix summary ---")
print("shape:", pointcare_matrix.shape, "dtype:", pointcare_matrix.dtype)
print("density:", float(pointcare_matrix.mean()))


=== SAMPLE WINDOW ===
file: day_custom_segments_001.csv
window_id: 0
rr_count (after 300–2000ms filter): 155
pointcare nonzero cells: 71
pointcare mapped points: 154

--- Feature engineering output (1 row) ---
heart_rate_bpm      77.951673
hf_power          4578.781254
label                0.000000
lf_hf_ratio          1.798981
lf_power          8237.139297
mean_rr_ms         769.707661
pnn50               44.805195
rmssd_ms            72.009806
sdnn_ms             86.083045
dtype: float64

--- Pointcaré matrix summary ---
shape: (30, 30) dtype: uint8
density: 0.07888888888888888


In [3]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Chọn nguồn label để benchmark:
# - "concatenated": dùng cột label trong data/concatenated/day_scenarios
# - "raw_gen": dùng tên thư mục con data/raw_gen/{0,1,2,3,...} làm label
DATA_MODE = "raw_gen"

RAW_GEN_ROOT = Path(r"C:/Users/buck/Napplee/StressClassification/data/raw_gen")
CONCAT_ROOT = RAW_FOLDER


def _require_symbols(symbols):
    missing = [s for s in symbols if s not in globals()]
    if missing:
        raise RuntimeError(
            "Thiếu các hàm/biến sau: " + ", ".join(missing) +
            ". Hãy chạy cell so sánh 1-window ngay phía trên trước." 
        )


_require_symbols([
    "extract_rr_ms_from_window",
    "build_pointcare_matrix",
    "feature_vector_from_window",
    "GRID_SIZE",
    "RR_MIN_MS_POINTCARE",
    "RR_MAX_MS_POINTCARE",
    "WINDOW_SEC",
])


def iter_labeled_csvs(data_mode: str):
    if data_mode == "concatenated":
        for csv_path in sorted(CONCAT_ROOT.glob("*.csv")):
            yield csv_path, None  # label lấy từ cột label trong file
        return

    if data_mode == "raw_gen":
        if not RAW_GEN_ROOT.exists():
            raise FileNotFoundError(f"Không thấy {RAW_GEN_ROOT}")

        for label_dir in sorted(RAW_GEN_ROOT.iterdir()):
            if not label_dir.is_dir():
                continue
            # label = tên folder, ví dụ "0", "1", "2"...
            try:
                label = int(label_dir.name)
            except Exception:
                continue

            for csv_path in sorted(label_dir.glob("*.csv")):
                yield csv_path, label
        return

    raise ValueError("DATA_MODE phải là 'concatenated' hoặc 'raw_gen'")


def build_datasets(
    data_mode: str,
    window_sec: float = 120.0,
    max_windows_per_file: int = 40,
):
    rows_feat = []
    X_point = []
    y = []
    groups = []

    for csv_path, label_from_dir in iter_labeled_csvs(data_mode):
        try:
            df = pd.read_csv(csv_path)
        except Exception as ex:
            print(f"Skip {csv_path}: read error {ex}")
            continue

        if not {"Time", "Voltage", "Peak"}.issubset(df.columns):
            continue

        df = df.sort_values("Time").reset_index(drop=True)
        min_time = df["Time"].min()
        max_time = df["Time"].max()
        n_windows = int(np.ceil((max_time - min_time) / window_sec))

        kept = 0
        for window_id in range(n_windows):
            if kept >= max_windows_per_file:
                break

            start = min_time + window_id * window_sec
            end = start + window_sec
            win = df[(df["Time"] >= start) & (df["Time"] < end)].copy()
            if win.empty:
                continue

            # label
            if data_mode == "concatenated":
                if "label" not in win.columns or win["label"].isnull().all():
                    continue
                label_val = win["label"].mode().iloc[0]
            else:
                label_val = label_from_dir

            if pd.isna(label_val):
                continue

            # RR + pointcare
            rr_ms = extract_rr_ms_from_window(win)
            if len(rr_ms) < 10:
                continue

            pointcare, _ = build_pointcare_matrix(
                rr_ms,
                grid_size=GRID_SIZE,
                rr_min_ms=RR_MIN_MS_POINTCARE,
                rr_max_ms=RR_MAX_MS_POINTCARE,
            )
            X_point.append(pointcare.reshape(-1).astype(float))

            # feature row
            feat = feature_vector_from_window(win)
            if feat is None:
                continue

            feat = dict(feat)
            feat["label"] = label_val
            rows_feat.append(feat)

            y.append(label_val)
            # group theo file để tránh leakage
            groups.append(str(csv_path))
            kept += 1

    df_feat = pd.DataFrame(rows_feat)
    X_point = np.asarray(X_point)
    y = pd.Series(y)
    groups = np.asarray(groups)

    feature_cols = [
        c for c in df_feat.columns
        if c not in {
            "label",
            "source_file",
            "window_id",
            "window_start",
            "window_end",
            "start_time",
            "end_time",
        }
    ]
    X_feat = df_feat[feature_cols].copy().apply(pd.to_numeric, errors="coerce")

    return X_feat, X_point, y, groups, feature_cols


X_feat, X_point, y, groups, feature_cols = build_datasets(
    DATA_MODE,
    window_sec=WINDOW_SEC,
    max_windows_per_file=40,
)

print("Dataset sizes:")
print("- Feature engineering:", X_feat.shape)
print("- Pointcaré flattened:", X_point.shape)
print("- #groups(files):", len(set(groups)))
print("Label distribution:")
print(y.value_counts(dropna=False))

if y.nunique() < 2:
    print("\nKhông thể train model vì label chỉ có 1 lớp.")
    print("- Nếu bạn đang để DATA_MODE='concatenated' thì có thể file day_scenarios chỉ chứa 1 nhãn.")
    print("- Hãy thử DATA_MODE='raw_gen' (label theo thư mục 0/1/2/3), hoặc kiểm tra lại cột label.")
    raise SystemExit(0)


gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_feat, y, groups=groups))

Xf_train, Xf_test = X_feat.iloc[train_idx], X_feat.iloc[test_idx]
Xp_train, Xp_test = X_point[train_idx], X_point[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


model_feat = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
model_feat.fit(Xf_train, y_train)
pred_f = model_feat.predict(Xf_test)


model_point = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
model_point.fit(Xp_train, y_train)
pred_p = model_point.predict(Xp_test)


def _report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1w = f1_score(y_true, y_pred, average="weighted")
    f1m = f1_score(y_true, y_pred, average="macro")
    print(f"\n{name}")
    print("accuracy:", round(acc, 4))
    print("f1_weighted:", round(f1w, 4))
    print("f1_macro:", round(f1m, 4))


_report("A) Feature engineering + LogisticRegression", y_test, pred_f)
_report("B) Pointcaré(30x30) flatten + LogisticRegression", y_test, pred_p)

print("\nGợi ý đọc kết quả:")
print("- A thường hợp với dataset nhỏ + sklearn (ít dữ liệu).")
print("- B (Pointcaré) thường hợp nhất khi dùng CNN/Deep Learning, hoặc khi RR pattern hình học rất rõ.")
print("- Nếu data/label chưa sạch (Peak sai, label lệch), cả 2 sẽ đều kém → ưu tiên cải thiện chất lượng dữ liệu trước.")


Dataset sizes:
- Feature engineering: (660, 8)
- Pointcaré flattened: (660, 900)
- #groups(files): 220
Label distribution:
3    186
0    162
1    162
2    150
Name: count, dtype: int64

A) Feature engineering + LogisticRegression
accuracy: 0.5303
f1_weighted: 0.5048
f1_macro: 0.5281

B) Pointcaré(30x30) flatten + LogisticRegression
accuracy: 0.5379
f1_weighted: 0.5351
f1_macro: 0.5418

Gợi ý đọc kết quả:
- A thường hợp với dataset nhỏ + sklearn (ít dữ liệu).
- B (Pointcaré) thường hợp nhất khi dùng CNN/Deep Learning, hoặc khi RR pattern hình học rất rõ.
- Nếu data/label chưa sạch (Peak sai, label lệch), cả 2 sẽ đều kém → ưu tiên cải thiện chất lượng dữ liệu trước.


In [3]:
# ===== Re-test Pointcaré on day_scenarios with WINDOW_SEC = 1800 (30 minutes) =====
# Cell này TỰ ĐỦ (không phụ thuộc cell phía trên) để bạn có thể run ngay.

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CONCAT_ROOT = Path(r"C:/Users/buck/Napplee/StressClassification/data/concatenated/day_scenarios")
WINDOW_SEC_DAY_SCENARIOS = 1800.0
GRID_SIZE = 30
RR_MIN_MS_POINTCARE = 400.0
RR_MAX_MS_POINTCARE = 1400.0


def extract_rr_ms_from_window(window_df, rr_min_ms=300.0, rr_max_ms=2000.0):
    window_df = window_df.sort_values("Time").drop_duplicates("Time")
    window_df = window_df[window_df["Time"].notna() & (window_df["Time"] >= 0)]
    window_df["Peak"] = pd.to_numeric(window_df["Peak"], errors="coerce")

    peak_times = pd.to_numeric(window_df.loc[window_df["Peak"] == 3, "Time"], errors="coerce").dropna().values
    if len(peak_times) < 2:
        return np.array([], dtype=float)

    rr_raw = np.diff(peak_times) * 1000.0
    rr_ms = rr_raw[(rr_raw >= rr_min_ms) & (rr_raw <= rr_max_ms)]
    return rr_ms


def build_pointcare_matrix(rr_ms, grid_size=30, rr_min_ms=400.0, rr_max_ms=1400.0):
    rr_ms = np.asarray(rr_ms, dtype=float)
    rr_ms = rr_ms[np.isfinite(rr_ms)]
    rr_ms = rr_ms[(rr_ms >= rr_min_ms) & (rr_ms <= rr_max_ms)]

    matrix = np.zeros((grid_size, grid_size), dtype=np.uint8)
    if len(rr_ms) < 2:
        return matrix, 0

    bins = np.linspace(rr_min_ms, rr_max_ms, grid_size + 1)
    x_idx = np.digitize(rr_ms[:-1], bins, right=False) - 1
    y_idx = np.digitize(rr_ms[1:], bins, right=False) - 1
    valid = (x_idx >= 0) & (x_idx < grid_size) & (y_idx >= 0) & (y_idx < grid_size)
    matrix[y_idx[valid], x_idx[valid]] = 1
    return matrix, int(valid.sum())


def _report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1w = f1_score(y_true, y_pred, average="weighted")
    f1m = f1_score(y_true, y_pred, average="macro")
    print(f"\n{name}")
    print("accuracy:", round(acc, 4))
    print("f1_weighted:", round(f1w, 4))
    print("f1_macro:", round(f1m, 4))


# 1) Raw row-level label distribution (đếm theo từng dòng trong CSV)
raw_label_counts = []
for csv_path in sorted(CONCAT_ROOT.glob("*.csv")):
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        continue
    if "label" not in df.columns:
        continue
    vc = df["label"].value_counts(dropna=False)
    for k, v in vc.items():
        raw_label_counts.append({"label": k, "count": int(v)})

print("=== Raw row-level label distribution (day_scenarios) ===")
raw_label_df = pd.DataFrame(raw_label_counts)
if raw_label_df.empty:
    print("Không thấy cột label trong các file day_scenarios")
else:
    print(raw_label_df.groupby("label")["count"].sum().sort_values(ascending=False))


# 2) Window-level (30 phút) majority label + build Pointcaré dataset
X_point = []
y = []
groups = []

for csv_path in sorted(CONCAT_ROOT.glob("*.csv")):
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        continue

    if not {"Time", "Voltage", "Peak", "label"}.issubset(df.columns):
        continue

    df = df.sort_values("Time").reset_index(drop=True)
    min_time = df["Time"].min()
    max_time = df["Time"].max()
    n_windows = int(np.ceil((max_time - min_time) / WINDOW_SEC_DAY_SCENARIOS))

    for window_id in range(n_windows):
        start = min_time + window_id * WINDOW_SEC_DAY_SCENARIOS
        end = start + WINDOW_SEC_DAY_SCENARIOS
        win = df[(df["Time"] >= start) & (df["Time"] < end)].copy()
        if win.empty:
            continue

        label_val = win["label"].mode().iloc[0] if not win["label"].isnull().all() else np.nan
        if pd.isna(label_val):
            continue

        rr_ms = extract_rr_ms_from_window(win)
        if len(rr_ms) < 10:
            continue

        pointcare, _ = build_pointcare_matrix(
            rr_ms,
            grid_size=GRID_SIZE,
            rr_min_ms=RR_MIN_MS_POINTCARE,
            rr_max_ms=RR_MAX_MS_POINTCARE,
        )

        X_point.append(pointcare.reshape(-1).astype(float))
        y.append(label_val)
        groups.append(csv_path.name)

X_point = np.asarray(X_point)
y = pd.Series(y)
groups = np.asarray(groups)

print("\n=== Window-level label distribution (30-min windows, majority) ===")
print(y.value_counts(dropna=False))
print("#windows:", len(y), "#files(groups):", len(set(groups)))

if len(y) == 0:
    raise ValueError("Không tạo được window nào (có thể RR quá ít hoặc cột Peak/label không đúng).")

if y.nunique() < 2:
    print("\nKết luận: Sau khi chia window 30 phút, label vẫn chỉ có 1 lớp → không train/test được.")
    print("Gợi ý: thử WINDOW_SEC nhỏ hơn (vd 120/300/600), hoặc kiểm tra lại cách gán label trong day_scenarios.")
else:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr, te = next(gss.split(X_point, y, groups=groups))

    Xp_tr, Xp_te = X_point[tr], X_point[te]
    y_tr, y_te = y.iloc[tr], y.iloc[te]

    model_point_30m = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])
    model_point_30m.fit(Xp_tr, y_tr)
    pred = model_point_30m.predict(Xp_te)

    print("\n=== Pointcaré benchmark on day_scenarios (30-min windows) ===")
    _report("Pointcaré(30x30) flatten + LogisticRegression", y_te, pred)
    print("\nClassification report:")
    print(classification_report(y_te, pred))


=== Raw row-level label distribution (day_scenarios) ===
label
0    98612756
1    58983360
2    35021374
3    28570070
Name: count, dtype: int64

=== Window-level label distribution (30-min windows, majority) ===
0    218
1    124
2     76
3     62
Name: count, dtype: int64
#windows: 480 #files(groups): 10

=== Pointcaré benchmark on day_scenarios (30-min windows) ===

Pointcaré(30x30) flatten + LogisticRegression
accuracy: 0.6979
f1_weighted: 0.7043
f1_macro: 0.6297

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.76      0.80        46
           1       0.73      0.70      0.71        23
           2       0.60      0.71      0.65        17
           3       0.33      0.40      0.36        10

    accuracy                           0.70        96
   macro avg       0.62      0.64      0.63        96
weighted avg       0.71      0.70      0.70        96



In [4]:
# ===== Export Pointcaré dataset (day_scenarios, WINDOW_SEC=1800) for train_part2_modeling =====
# Chạy cell ngay phía trên trước (để có extract_rr_ms_from_window/build_pointcare_matrix).

from pathlib import Path
import numpy as np
import pandas as pd

EXPORT_DIR = Path(r"C:/Users/buck/Napplee/StressClassification/data/features/pointcare")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

OUT_BASENAME = f"day_scenarios_w{int(WINDOW_SEC_DAY_SCENARIOS)}_grid{GRID_SIZE}_rr{int(RR_MIN_MS_POINTCARE)}to{int(RR_MAX_MS_POINTCARE)}"
OUT_NPZ = EXPORT_DIR / f"{OUT_BASENAME}.npz"
OUT_META_CSV = EXPORT_DIR / f"{OUT_BASENAME}_meta.csv"
OUT_WIDE_CSV = EXPORT_DIR / f"{OUT_BASENAME}_wide.csv"

meta_rows = []
X_rows = []

for csv_path in sorted(CONCAT_ROOT.glob("*.csv")):
    try:
        df = pd.read_csv(csv_path)
    except Exception:
        continue

    if not {"Time", "Voltage", "Peak", "label"}.issubset(df.columns):
        continue

    df = df.sort_values("Time").reset_index(drop=True)
    min_time = df["Time"].min()
    max_time = df["Time"].max()
    n_windows = int(np.ceil((max_time - min_time) / WINDOW_SEC_DAY_SCENARIOS))

    for window_id in range(n_windows):
        start = float(min_time + window_id * WINDOW_SEC_DAY_SCENARIOS)
        end = float(start + WINDOW_SEC_DAY_SCENARIOS)
        win = df[(df["Time"] >= start) & (df["Time"] < end)].copy()
        if win.empty:
            continue

        label_val = win["label"].mode().iloc[0] if not win["label"].isnull().all() else np.nan
        if pd.isna(label_val):
            continue

        rr_ms = extract_rr_ms_from_window(win)
        if len(rr_ms) < 10:
            continue

        pointcare, _ = build_pointcare_matrix(
            rr_ms,
            grid_size=GRID_SIZE,
            rr_min_ms=RR_MIN_MS_POINTCARE,
            rr_max_ms=RR_MAX_MS_POINTCARE,
        )

        meta_rows.append({
            "source_file": csv_path.name,
            "window_id": int(window_id),
            "start_time": start,
            "end_time": end,
            "label": int(label_val),
        })
        X_rows.append(pointcare.reshape(-1).astype(np.uint8))

if not meta_rows:
    raise ValueError("Không export được window nào. Kiểm tra lại Peak==3, label, hoặc WINDOW_SEC.")

meta_df = pd.DataFrame(meta_rows)
X = np.stack(X_rows, axis=0).astype(np.uint8)
y_arr = meta_df["label"].to_numpy().astype(int)
groups_arr = meta_df["source_file"].to_numpy().astype(str)

# Save compact format for fast loading
np.savez_compressed(OUT_NPZ, X=X, y=y_arr, groups=groups_arr)
meta_df.to_csv(OUT_META_CSV, index=False)

# Optional: wide CSV (pc_0..pc_899) for quick inspection / pandas workflows
pc_cols = [f"pc_{i}" for i in range(X.shape[1])]
wide_df = pd.concat([meta_df.reset_index(drop=True), pd.DataFrame(X, columns=pc_cols)], axis=1)
wide_df.to_csv(OUT_WIDE_CSV, index=False)

print("=== Exported Pointcaré dataset ===")
print("X shape:", X.shape, "dtype:", X.dtype)
print("Label distribution:")
print(pd.Series(y_arr).value_counts().sort_index())
print("Unique groups(source_file):", pd.Series(groups_arr).nunique())
print("Saved:")
print("-", OUT_NPZ)
print("-", OUT_META_CSV)
print("-", OUT_WIDE_CSV)

=== Exported Pointcaré dataset ===
X shape: (480, 900) dtype: uint8
Label distribution:
0    218
1    124
2     76
3     62
Name: count, dtype: int64
Unique groups(source_file): 10
Saved:
- C:\Users\buck\Napplee\StressClassification\data\features\pointcare\day_scenarios_w1800_grid30_rr400to1400.npz
- C:\Users\buck\Napplee\StressClassification\data\features\pointcare\day_scenarios_w1800_grid30_rr400to1400_meta.csv
- C:\Users\buck\Napplee\StressClassification\data\features\pointcare\day_scenarios_w1800_grid30_rr400to1400_wide.csv
